In [10]:
from docling.document_converter import DocumentConverter

source = "/Users/karim/Desktop/laws for chatbot/trade.pdf"
converter = DocumentConverter()
doc_trade = converter.convert(source)
md_trade = doc_trade.document.export_to_markdown()

lines = md_trade.split("\n")
print(f"Total lines: {len(lines)}")
print()
for i, line in enumerate(lines[:100]):
    if line.strip():
        print(f"{i}: {repr(line.strip())}")

Total lines: 1492

0: 'The Federal Assembly of the Czech and Slovak Federative Republic has passed the following Act:'
2: '## GENERAL PROVISIONS'
4: '## TITLE I'
6: '## SUBJECT OF REGULATION'
8: 'Section 1'
10: "This Act lays down conditions for carrying on a licensed trade (hereinafter referred to as ' trade ' ) and inspections of compliance with those conditions."
12: '## Trades'
14: 'Section 2'
16: "A  trade  shall  mean  a  systematic  activity  carried  out  independently  under  the  conditions  laid down in this Act, under a person's own name and liability, with a view to making a profit."
18: '## Section 3'
20: '(1) The following shall not constitute a trade:'
22: '- a) the performance of an activity statutorily reserved for the State or for a designated legal person,'
23: '- b) the  use  of  the  results  of  intellectual  creativity  protected  by  specific  laws,  their  originators  or authors 2) ,'
24: '- c) the  collective  administration  of  copyright  and  rights  rela

In [11]:
import re
section_lines = [(i, repr(line.strip())[:120]) for i, line in enumerate(lines) if re.search(r"Section\s+\d", line)]
print(f"Total section-like lines: {len(section_lines)}")
print("\nFirst 15:")
for idx, txt in section_lines[:15]:
    print(f"  Line {idx}: {txt}")
print("\nLast 10:")
for idx, txt in section_lines[-10:]:
    print(f"  Line {idx}: {txt}")

Total section-like lines: 246

First 15:
  Line 8: 'Section 1'
  Line 14: 'Section 2'
  Line 18: '## Section 3'
  Line 34: '2c) Section 21(2) of Act No 20/1987 Coll., on the care of monuments by the State.'
  Line 72: '9) Section  11  and  Section  13(1)  of  Act  No  2/1991  Coll.,  on  collective  bargaining,  as  amended  by  Act  No
  Line 78: '10a) Section 14(1)(a) of Act No 360/1992 Coll., on the profession of authorised architects and the profession of author
  Line 80: '10b) Section 144(4) of Act No 183/2006 Coll., on land-use planning and building rules (the Building Act).'
  Line 134: '18) Section 60(3) of Act No 266/1994 Coll., on railways.'
  Line 173: '23h) Section 27 of Act No 250/2000 Coll., on budgetary rules of territorial budgets.'
  Line 175: '23i) Section 4(2)(b) and Sections 48 to 50 of Act No 359/1999 Coll., on child protection.'
  Line 206: '56) Section  81  of  Act  No  326/2004  Coll.,  on  phytosanitary  care  and  amending  certain  related  acts,  as ame
  L

In [12]:
from collections import OrderedDict

def preprocess_trade_act(text):
    new_lines = []
    for line in text.split("\n"):
        stripped = line.strip()

        if re.match(r"^\d+[a-z]?\)\s", stripped):
            continue

        parts = re.split(r"(Section\s+\d+)", line, flags=re.IGNORECASE)
        if len(parts) <= 2:
            new_lines.append(line)
        else:
            current = parts[0]
            for i in range(1, len(parts)):
                if re.match(r"Section\s+\d+", parts[i], re.IGNORECASE):
                    if current.strip():
                        new_lines.append(current)
                    current = parts[i]
                else:
                    current += parts[i]
            if current.strip():
                new_lines.append(current)

    return "\n".join(new_lines)


def parse_trade_act(text, law_name="Trade Licensing Act (455/1991)"):
    text = preprocess_trade_act(text)

    section_pattern = re.compile(
        r"^(?:#+\s*)?(?:-\s*)?Section\s+(\d+)\s*(?:\[.*?\])?\s*(.*)",
        re.IGNORECASE
    )
    title_pattern = re.compile(r"^(?:#+\s*)?TITLE\s+([IVXLC]+)", re.IGNORECASE)
    part_pattern = re.compile(r"^(?:#+\s*)?PART\s+(\w+)", re.IGNORECASE)
    division_pattern = re.compile(r"^(?:#+\s*)?Division\s+(\d+)", re.IGNORECASE)
    skip_patterns = [
        re.compile(r"^#{1,6}\s*$"),
        re.compile(r"^-\s*\d+\s*-\s*$"), 
    ]

    current_part_id = None
    current_part_name = None
    current_title_id = None
    current_title_name = None
    current_division_id = None
    current_division_name = None
    expecting_part_name = False
    expecting_title_name = False
    expecting_division_name = False
    current_section_id = None
    current_text_lines = []
    all_chunks = []

    def clean_md(line):
        return re.sub(r"^#+\s*", "", line).strip()

    def should_skip(line):
        for p in skip_patterns:
            if p.match(line):
                return True
        return False

    def save_chunk():
        if current_section_id is None:
            return
        text_joined = " ".join(current_text_lines).strip()

        
        if not text_joined or text_joined.lower() == "(deleted)" or text_joined.lower() == "deleted":
            return

        all_chunks.append({
            "law_name": law_name,
            "part_id": current_part_id or "Unknown",
            "part_name": current_part_name or "Unknown",
            "title_id": current_title_id or "Unknown",
            "title_name": current_title_name or "Unknown",
            "division_id": current_division_id or "None",
            "division_name": current_division_name or "None",
            "section_id": current_section_id,
            "text": text_joined,
        })

    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue

        if re.match(r"^\d+[a-z]?\)\s", stripped):
            continue

        section_match = section_pattern.match(stripped)
        if section_match:
            save_chunk()
            current_section_id = f"Section {section_match.group(1)}"
            current_text_lines = []
            expecting_part_name = False
            expecting_title_name = False
            expecting_division_name = False
            remaining = section_match.group(2).strip()
            if remaining:
                current_text_lines.append(remaining)
            continue

        part_match = part_pattern.match(stripped)
        if part_match:
            save_chunk()
            current_section_id = None
            current_text_lines = []
            current_part_id = f"Part {part_match.group(1)}"
            current_part_name = None
            current_title_id = None
            current_title_name = None
            current_division_id = None
            current_division_name = None
            expecting_part_name = True
            expecting_title_name = False
            expecting_division_name = False
            continue

        title_match = title_pattern.match(stripped)
        if title_match:
            current_title_id = f"Title {title_match.group(1)}"
            current_title_name = None
            current_division_id = None
            current_division_name = None
            expecting_title_name = True
            expecting_part_name = False
            expecting_division_name = False
            continue

        division_match = division_pattern.match(stripped)
        if division_match:
            current_division_id = f"Division {division_match.group(1)}"
            current_division_name = None
            expecting_division_name = True
            expecting_part_name = False
            expecting_title_name = False
            continue

        if should_skip(stripped):
            continue

        clean = clean_md(stripped)
        if not clean:
            continue

        if expecting_part_name:
            current_part_name = clean
            expecting_part_name = False
            continue

        if expecting_title_name:
            current_title_name = clean
            expecting_title_name = False
            continue

        if expecting_division_name:
            current_division_name = clean
            expecting_division_name = False
            continue

        if current_section_id is not None:
            current_text_lines.append(stripped)

    save_chunk()

    for chunk in all_chunks:
        t = chunk["text"]
        t = re.sub(r"\s{2,}", " ", t)
        chunk["text"] = t.strip()

    merged = OrderedDict()
    for c in all_chunks:
        sid = c["section_id"]
        if sid not in merged:
            merged[sid] = c.copy()
        else:
            existing = merged[sid]["text"]
            new_text = c["text"]
            if new_text not in existing:
                merged[sid]["text"] = existing + " " + new_text

    chunks = sorted(merged.values(), key=lambda c: int(c["section_id"].split()[-1]))
    return chunks


chunks_trade = parse_trade_act(md_trade)
print(f"Total sections parsed: {len(chunks_trade)}")


all_nums = sorted([int(c["section_id"].split()[-1]) for c in chunks_trade])
missing = [i for i in range(all_nums[0], all_nums[-1] + 1) if i not in all_nums]

print(f"Section range: {all_nums[0]} — {all_nums[-1]}")
print(f"Missing sections: {len(missing)}")

if missing:
    print(f"Missing: {missing}")

deleted_in_pdf = [4, 6, 7, 12, 15, 16, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 51, 59, 65, 66, 69]
print(f"\nKnown deleted sections: {len(deleted_in_pdf)}")
actual_missing = [m for m in missing if m not in deleted_in_pdf]
print(f"Unexpectedly missing (not deleted): {actual_missing if actual_missing else 'NONE'}")

print(f"\n--- Spot checks ---")
for n in [1, 2, 3, 5, 6, 8, 17, 31, 45, 60, 73, 81]:
    found = [c for c in chunks_trade if c["section_id"] == f"Section {n}"]
    if found:
        c = found[0]
        print(f"Section {n}: {c['part_id']} / {c['title_id']} | {c['text'][:70]}...")
    else:
        print(f"Section {n}: MISSING (likely deleted)")


Total sections parsed: 75
Section range: 1 — 81
Missing sections: 6
Missing: [15, 29, 39, 40, 43, 66]

Known deleted sections: 23
Unexpectedly missing (not deleted): [43]

--- Spot checks ---
Section 1: Unknown / Title I | This Act lays down conditions for carrying on a licensed trade (herein...
Section 2: Unknown / Title I | A trade shall mean a systematic activity carried out independently und...
Section 3: Unknown / Title I | (1) The following shall not constitute a trade: - a) the performance o...
Section 5: Unknown / Title II | ## Entities eligible to carry on a trade (1) A natural or legal person...
Section 6: Unknown / Title II | ## General conditions for carrying on a trade (1) Unless otherwise pro...
Section 8: Unknown / Title II | ## Impediments to carrying on a trade (1) A natural or legal person wh...
Section 17: Unknown / Title II | (1) For the purposes of this Act, an establishment shall mean the spac...
Section 31: Unknown / Title II | (11)). (9) An entrepreneur may also

In [13]:
import chromadb
client_trade = chromadb.PersistentClient(path="./data/law_db")
collection_trade = client_trade.get_or_create_collection(
    name="czech_trade_law",
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection ready: {collection_trade.count()} items")

Collection ready: 0 items


In [14]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
texts_trade = [c["text"] for c in chunks_trade]
embeddings_trade = model.encode(texts_trade)
print(embeddings_trade.shape)

(75, 384)


In [15]:
# CELL 3: Add to ChromaDB
all_ids = []
all_documents = []
all_metadatas = []
all_embeddings = embeddings_trade.tolist()

for i, chunk in enumerate(chunks_trade):
    all_ids.append(f"trade_{i}")
    all_documents.append(chunk["text"])
    all_metadatas.append({
        "law_name":      chunk.get("law_name", "Unknown") or "Unknown",
        "part_id":       chunk.get("part_id", "Unknown") or "Unknown",
        "part_name":     chunk.get("part_name", "Unknown") or "Unknown",
        "title_id":      chunk.get("title_id", "Unknown") or "Unknown",
        "title_name":    chunk.get("title_name", "Unknown") or "Unknown",
        "division_id":   chunk.get("division_id", "Unknown") or "Unknown",
        "division_name": chunk.get("division_name", "Unknown") or "Unknown",
        "section_id":    chunk.get("section_id", "Unknown") or "Unknown",
    })

batch_size = 50
for start in range(0, len(all_ids), batch_size):
    end = start + batch_size
    collection_trade.add(
        ids=all_ids[start:end],
        documents=all_documents[start:end],
        metadatas=all_metadatas[start:end],
        embeddings=all_embeddings[start:end],
    )
    done = min(end, len(all_ids))
    print(f"Added {done}/{len(all_ids)}")

print(f"\nDone! Collection has {collection_trade.count()} items")

Added 50/75
Added 75/75

Done! Collection has 75 items
